In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import zipfile
import re
!pip install biosppy
import biosppy
import biosppy.signals.ecg as ecg
from scipy import signal
from IPython.display import clear_output
!pip install hrv-analysis
import hrvanalysis
from hrvanalysis import get_time_domain_features, get_frequency_domain_features

In [ ]:
# import os
# import zipfile
# for root, dirs, files in os.walk(".", topdown=False):
#    for name in files:
#         file_path = os.path.join(root, name)
#         if 'FilteredECG' in file_path and '.zip' in file_path:
#             folder_path = os.path.split(file_path)[0]
# #             print(file_path)
#             # Extract the contents of the zip file to the same folder
#             with zipfile.ZipFile(file_path, 'r') as zip_ref:
#                 zip_ref.extractall(folder_path)
# #             Optional: remove the zip file after extraction
#             os.remove(file_path)

In [ ]:
dummyf = []
list_names = []
for root, dirs, files in os.walk("."):
    if os.path.split(root)[1] == '250' or  os.path.split(root)[1] == '500':
        root = root[2:] # remove .  
        dummyf.append(root)     
        head, tail = os.path.split(root)
        head, tail = os.path.split(head)
        a, b = os.path.split(head)
        a, c = os.path.split(a)
        list_names.append(a + "_" + b)

dummyf = dummyf[30:33]
list_names = list_names[30:33]
print(list_names)
print(dummyf)

In [ ]:
for i,name in zip(dummyf,list_names):
    parent_dir = i
    print(parent_dir)

    # read ecg data
    # Create an empty list to hold ECG data and time
    ecg_data = []

    # Loop over each file in the directory
    for filename in os.listdir(parent_dir):
        if filename.endswith(".txt"):
            # Get the path to the txt file
            file_path = os.path.join(parent_dir, filename)
            # Read the ECG data from the txt file
            with open(file_path, 'r') as file:
                ecg_raw = file.read()
            # Clean the ECG data
            ecg_clean = re.sub(r'[a-zA-Z]', '', ecg_raw)
            ecg_clean = ecg_clean.replace(",", "")
            ecg_float = np.array([np.float32(x) for x in ecg_clean.split()])
            # Determine the time interval for each data point
            time_duration = 600  # 10-minute interval
            time_step = time_duration / len(ecg_float)
            # Create a time array for the ECG data
            start_time = int(re.search(r'\d+', filename).group(0))
            time_array = np.linspace(start_time, start_time+time_duration, len(ecg_float))
            # Append the ECG data and time array to the list
            ecg_data.append(np.column_stack((time_array, ecg_float)))
    ecg_data_concat = np.concatenate(ecg_data, axis=0)
    df = pd.DataFrame(ecg_data_concat, columns=['time', 'ECG'])
    df['time'] = pd.to_datetime(df['time'], unit='s')
    
    # PLOTTING ECG AGAINST TIME TO VISUALISE ECG DATA
    # Select 6s and 1 hour of data
    start_time = df['time'].iloc[0]
    end_time = start_time + pd.Timedelta(minutes=0.1)
    end_time1 = start_time + pd.Timedelta(minutes=60)
    df_6s = df[(df['time'] >= start_time) & (df['time'] < end_time)]
    df_hour = df[(df['time'] >= start_time) & (df['time'] < end_time1)]

    # Plot ECG data
    plt.plot(df_6s['time'], df_6s['ECG'])
    plt.xlabel('Time')
    plt.ylabel('ECG')
    plt.title('ECG Data for 6 seconds')
    plt.show()

    # Plot ECG data
    plt.plot(df_hour['time'], df_hour['ECG'])
    plt.xlabel('Time')
    plt.ylabel('ECG')
    plt.title('ECG Data for an hour')
    plt.show()
    
    # get time and freq domain in 30s windows 1s roll

    window_size_sec = 30
    step_sec = 1

    # Convert window size and step size in seconds to number of data points
    window_size = int(window_size_sec * 250) # 250 is the sampling rate
    step = int(step_sec * 250) # 250 is the sampling rate

    df['unixTime'] = 0

    # Convert datetime format to Unix timestamps
    df['unixTime'] = (df['time'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

    # calculate the time difference between consecutive rows of the 'time' column in seconds
    df['time_sec'] = (df['unixTime'] - df['unixTime'].shift())
    df['time_sec'].iloc[0]=0

    # initialize dictionaries to store time and frequency domain features
    time_domain_features_dict = {}
    frequency_domain_features_dict = {}
    filtered_ecg_dict = {}
    df_time = pd.DataFrame(columns=['start_time', 'end_time'])
    count = 0

    # iterate over the windows with the specified overlap
    for i in range(0,len(df)-window_size+1, step):
        try:
    #         print(f"{i=}")
            # extract the window
            window = df.iloc[i:i+window_size]
            signal = list(window['ECG'])
            count = count+1
    #         print(f"{count=}")
            out = biosppy.signals.ecg.ecg(signal, 250, show=False)
            # Extract filtered ECG signal and R-peak locations
            filtered_ecg = out['filtered']
            rpeaks = out['rpeaks']
            rpeaks = list(rpeaks)

            # Assume rpeaks is a one-dimensional array of sample indices of the R-peaks
            rpeak_times = window['time'].iloc[rpeaks]

            # time intervals between the rpeaks
            rr_intervals = np.diff(rpeak_times)/1000000 # this is in ns
            rr_intervals = rr_intervals.tolist() #################

            # Shift rpeak_times array to create an array of times for each RR interval
            interval_times = rpeak_times[1:]
            nn_intervals = hrvanalysis.preprocessing.get_nn_intervals(rr_intervals, 300, 2000, 'inside', 'both', 'linear')
            nn_intervals = np.array(nn_intervals)  # convert to numpy array
            nn_intervals = nn_intervals[~np.isnan(nn_intervals)]

            # Add datetime column to df_both
            start_time = window['time'].iloc[0]
            end_time = window['time'].iloc[-1]
            df_time.loc[i, 'start_time'] = start_time
            df_time.loc[i, 'end_time'] = end_time

            time_domain_features = hrvanalysis.extract_features.get_time_domain_features(nn_intervals) 
            frequency_domain_features = hrvanalysis.extract_features.get_frequency_domain_features(nn_intervals, 'welch', int(len(window)/30), 'linear')
            if len(rr_intervals) < 15: # less than 15 beats every 30s
                time_domain_features = {}
                frequency_domain_features = {}

        except:
    #         print(f"Error {i}")
            # set time_domain_features and frequency_domain_features to empty dictionaries
            time_domain_features = {}
            frequency_domain_features = {}
        # check if the time domain features list is not empty
        if time_domain_features:
            # store time domain features as a dictionary in the 'time_domain_features_dict' dictionary
            time_domain_features_dict[i] = time_domain_features 

        # check if the frequency domain features list is not empty
        if frequency_domain_features:
            # store frequency domain features as a dictionary in the 'frequency_domain_features_dict' dictionary
            frequency_domain_features_dict[i] = frequency_domain_features

    # create a DataFrame for the time domain features
    time_domain_df = pd.DataFrame.from_dict(time_domain_features_dict, orient='index')

    # create a DataFrame for the frequency domain features
    frequency_domain_df = pd.DataFrame.from_dict(frequency_domain_features_dict, orient='index')

    df_domains = pd.concat([time_domain_df, frequency_domain_df], axis=1)
    #######################################
    # Convert time columns to datetime format
    df_time['start_time'] = pd.to_datetime(df_time['start_time'])
    df_time['end_time'] = pd.to_datetime(df_time['end_time'])
    # Convert datetimes to Unix timestamps
    df_time['start_unix'] = (pd.to_datetime(df_time['start_time'], format='%Y-%m-%d %H:%M:%S', utc=True) - pd.Timestamp("1970-01-01", tz='UTC')) // pd.Timedelta('1s')
    df_time['end_unix'] = (pd.to_datetime(df_time['end_time'], format='%Y-%m-%d %H:%M:%S', utc=True) - pd.Timestamp("1970-01-01", tz='UTC')) // pd.Timedelta('1s')

    #######################################

    # merge the time domain and frequency domain DataFrames
    df_both = pd.concat([df_time, time_domain_df, frequency_domain_df], axis=1)
    
    clear_output()
    print(df_both)
    df_both.to_csv(name + '.csv')